## Verify files in relevant volume

In [0]:

file_path_xlsx = "/Volumes/bronze_dev/census_bureau/census_bureau_raw/co-est2025-pop processed.xlsx"
file_path_csv = file_path_xlsx.replace("xlsx","csv")

display(dbutils.fs.ls("/Volumes/bronze_dev/census_bureau/raw/"))


## Read csv with spark ✅

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path_csv)

df.head()

In [0]:
display(df)
#df.show(5)

In [0]:
from pyspark.sql.functions import col, split, regexp_replace

clean_col = regexp_replace(col("county"), r"^\.", "")

df_clean = df.withColumn("county_name", split(clean_col, ", ").getItem(0)) \
             .withColumn("state_name", split(clean_col, ", ").getItem(1)) \
             .drop("county")
display(df_clean)

In [0]:
df_final = df_clean.select(
    "county_name",
    "state_name",
    *[c for c in df_clean.columns if c not in ["county_name", "state_name"]]
)
display(df_final)

In [0]:
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_dev.census_bureau.county_pop")

In [0]:
%sql
SELECT * FROM bronze_dev.census_bureau.county_pop LIMIT 10;